In [5]:
import pandas as pd
import warnings
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

df = pd.read_excel("data_For_Analysis.csv")
df['Date'] = pd.to_datetime(df['Date'])

monthly = df.groupby(['Part', pd.Grouper(key='Date', freq='ME')])['Quantity'].sum().reset_index()

valid_parts = monthly.groupby('Part').filter(lambda x: len(x) >= 15)

forecast_results = []
accuracy_results = []

print("Total Parts:", valid_parts['Part'].nunique())

for part in valid_parts['Part'].unique():

    part_df = valid_parts[valid_parts['Part'] == part]
    ts = part_df.set_index('Date')['Quantity']
    ts = ts.asfreq('ME')

    train = ts[:-3]
    test = ts[-3:]

    try:
        arima_fit = ARIMA(train, order=(1,1,1)).fit()
        arima_pred = arima_fit.forecast(len(test))

        sarima_fit = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False)
        sarima_pred = sarima_fit.forecast(len(test))

        hw_fit = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
        hw_pred = hw_fit.forecast(len(test))

        ensemble_pred = (arima_pred + sarima_pred + hw_pred) / 3
        avg_pred = (arima_pred + sarima_pred + hw_pred + ensemble_pred) / 4

        mse_arima = mean_squared_error(test, arima_pred)
        rmse_arima = np.sqrt(mse_arima)

        mse_sarima = mean_squared_error(test, sarima_pred)
        rmse_sarima = np.sqrt(mse_sarima)

        mse_hw = mean_squared_error(test, hw_pred)
        rmse_hw = np.sqrt(mse_hw)

        mse_ens = mean_squared_error(test, ensemble_pred)
        rmse_ens = np.sqrt(mse_ens)

        mse_avg = mean_squared_error(test, avg_pred)
        rmse_avg = np.sqrt(mse_avg)

        accuracy_results.append({
            "Part": part,
            "ARIMA_MSE": mse_arima, "ARIMA_RMSE": rmse_arima,
            "SARIMA_MSE": mse_sarima, "SARIMA_RMSE": rmse_sarima,
            "HoltWinters_MSE": mse_hw, "HoltWinters_RMSE": rmse_hw,
            "Ensemble_MSE": mse_ens, "Ensemble_RMSE": rmse_ens,
            "AVG_MSE": mse_avg, "AVG_RMSE": rmse_avg
        })

        future_dates = pd.date_range(
            start=ts.index.max() + pd.offsets.MonthEnd(1),
            periods=6,
            freq='ME'
        )

        arima_f = ARIMA(ts, order=(1,1,1)).fit().forecast(6)
        sarima_f = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False).forecast(6)
        hw_f = ExponentialSmoothing(ts, trend='add', seasonal='add', seasonal_periods=12).fit().forecast(6)

        ensemble_f = (arima_f + sarima_f + hw_f) / 3
        avg_f = (arima_f + sarima_f + hw_f + ensemble_f) / 4

        temp_df = pd.DataFrame({
            "Part": part,
            "Date": future_dates,
            "ARIMA": arima_f.round(0).astype(int),
            "SARIMA": sarima_f.round(0).astype(int),
            "Holt_Winters": hw_f.round(0).astype(int),
            "Ensemble": ensemble_f.round(0).astype(int),
            "AVG": avg_f.round(0).astype(int)
        })

        forecast_results.append(temp_df)

    except:
        print(f"Skipping Part {part}")
        continue

forecast_df = pd.concat(forecast_results, ignore_index=True)
accuracy_df = pd.DataFrame(accuracy_results)

forecast_df.to_excel("forecast_with_models_avg.xlsx", index=False)
accuracy_df.to_excel("model_accuracy_mse_rmse.xlsx", index=False)

print("Forecast & Accuracy calculation completed successfully")
print("Forecast file: forecast_with_models_avg.xlsx")
print("Accuracy file: model_accuracy_mse_rmse.xlsx")

Total Parts: 76
Skipping Part 92000513
Skipping Part 92000514
Skipping Part 92000946
Skipping Part 92250385
Skipping Part 92250386
Skipping Part 92250395
Skipping Part 92250401
Skipping Part 94300314
Skipping Part 94600328
Skipping Part 94600330
Skipping Part 94600471
Skipping Part 94600472
Skipping Part 94600473
Skipping Part 94700357
Skipping Part 94700358
Skipping Part 94700359
Skipping Part 94900438
Skipping Part 94950063
Skipping Part 94950457
Skipping Part 94952121
Skipping Part 99260212
Skipping Part 99260276
Skipping Part 99300093
Skipping Part 99300095
Skipping Part 99300096
Skipping Part 99300098
Skipping Part 99300379
Skipping Part 99951012
Skipping Part 99951030
Skipping Part 99951187
Forecast & Accuracy calculation completed successfully
Forecast file: forecast_with_models_avg.xlsx
Accuracy file: model_accuracy_mse_rmse.xlsx
